# CYCLOPS SENTINEL: Benchmark de Algoritmos Supervisados

Evaluacion cuantitativa comparativa entre tres familias de algoritmos para la clasificacion autonoma de phishing: Arbol de Decision, K-Nearest Neighbors - KNN y Gaussian Naive Bayes.

## Metodologia de Evaluacion
1. Particion estratificada 80 a 20 sobre 91992 registros balanceados.
2. Acondicionamiento con RobustScaler ajustado estrictamente sobre el 80% de entrenamiento para evitar fuga de datos.
3. Cronometraje riguroso del tiempo de entrenamiento y latencia de prediccion por muestra.
4. Evaluacion independiente en el conjunto de prueba mediante Exactitud, Precision, Sensibilidad Recall, F1-Score y ROC-AUC.
5. Inteligencia Artificial Explicable - XAI mediante pesos de importancia de caracteristicas.


In [ ]:
# 1. Configuracion del Entorno y Librerias
import os
import time
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Segoe UI'
plt.rcParams['figure.dpi'] = 120

os.makedirs('reports/figures', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('[OK] Entorno del benchmark supervisado configurado con exito.')


## 2. Ingesta y Construccion de la Matriz de Caracteristicas

Se carga `clean_dataset.csv` y se genera la matriz de entrada con metricas de longitud, indicadores tecnicos binarios y densidades de urgencia y enlaces.


In [ ]:
# 2. Ingesta y Matriz de Variables
data_path = '../data/processed/clean_dataset.csv'
if not os.path.exists(data_path):
    data_path = r'C:/Users/Cristian Díaz/Documents/CYCLOPS/data/processed/clean_dataset.csv'

df = pd.read_csv(data_path)
base_features = ['char_count', 'word_count', 'url_count', 'has_url', 'has_ip', 'urgency_score']

X = df[base_features].copy()
X['url_density'] = (X['url_count'] / np.maximum(X['word_count'], 1)) * 100
X['urgency_density'] = (X['urgency_score'] / np.maximum(X['word_count'], 1)) * 100
y = df['is_threat']

print(f'Dimensiones del dataset: {df.shape[0]:,} filas y {X.shape[1]} columnas de entrada.')
print('Distribucion de la variable objetivo is_threat:')
for label_val, count_val in y.value_counts().items():
    pct = float(count_val) / float(len(y)) * 100.0
    tag = 'Amenaza Phishing' if label_val == 1 else 'Legitimo'
    print(f' - Clase {label_val} [{tag}]: {count_val:,} muestras - {pct:.2f}%')


## 3. Particion Estratificada 80 a 20 y Escalado Robusto

Para blindar el proceso contra fuga de datos, RobustScaler se ajusta unicamente con el conjunto de entrenamiento. Luego, ese transformador congelado escala tanto el 80% de entrenamiento como el 20% de prueba.


In [ ]:
# 3. Particion Estratificada y Ajuste de RobustScaler
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Conjunto de entrenamiento: {len(X_train):,} muestras - 80 por ciento')
print(f'Conjunto de evaluacion:    {len(X_test):,} muestras - 20 por ciento')
print('[OK] RobustScaler ajustado sin fuga de datos.')


## 4. Entrenamiento y Evaluacion Cronometrada de los Tres Modelos

Se entrenan y miden:
- **Arbol de Decision:** Particion jerarquica con criterio de entropia y poda a 15 niveles.
- **K-Nearest Neighbors - KNN:** Clasificador de distancias euclidianas con K=5 y estructura KDTree.
- **Gaussian Naive Bayes:** Modelo bayesiano asumiendo normalidad e independencia condicional.


In [ ]:
# 4. Entrenamiento y Medicion de Latencias
models_dict = {
    'Arbol de Decision': DecisionTreeClassifier(max_depth=15, criterion='entropy', random_state=42),
    'K-Nearest Neighbors - KNN': KNeighborsClassifier(n_neighbors=5, algorithm='kd_tree', n_jobs=-1),
    'Gaussian Naive Bayes': GaussianNB()
}

benchmark_results = []
eval_predictions = {}
eval_probabilities = {}

for name, model in models_dict.items():
    t_start = time.time()
    model.fit(X_train_scaled, y_train)
    t_train = time.time() - t_start
    
    t_start = time.time()
    preds = model.predict(X_test_scaled)
    t_pred = time.time() - t_start
    
    probs = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else preds
    
    eval_predictions[name] = preds
    eval_probabilities[name] = probs
    
    acc = float(accuracy_score(y_test, preds))
    prec = float(precision_score(y_test, preds, zero_division=0))
    rec = float(recall_score(y_test, preds, zero_division=0))
    f1 = float(f1_score(y_test, preds, zero_division=0))
    auc = float(roc_auc_score(y_test, probs))
    latency_ms = (t_pred / float(len(X_test))) * 1000.0
    
    benchmark_results.append({
        'Algoritmo': name,
        'Exactitud (%)': round(acc * 100, 2),
        'Precision (%)': round(prec * 100, 2),
        'Recall (%)': round(rec * 100, 2),
        'F1-Score (%)': round(f1 * 100, 2),
        'ROC-AUC (%)': round(auc * 100, 2),
        'Tiempo Train (s)': round(t_train, 4),
        'Latencia / Muestra (ms)': round(latency_ms, 4)
    })

results_df = pd.DataFrame(benchmark_results)
print('[OK] Todos los modelos entrenados y evaluados exitosamente.')


## 5. Tabla Comparativa Ejecutiva de Rendimiento

Matriz ejecutiva con metricas predictivas y costos de computo sobre el conjunto independiente de 18399 muestras.


In [ ]:
# 5. Tabla Resumen Ejecutiva
print('=== Resumen Comparativo de Algoritmos Supervisados ===')
print(results_df.to_string(index=False))


## 6. Matrices de Confusion y Curvas ROC

Visualizacion comparativa de verdaderos positivos, falsos positivos, verdaderos negativos y falsos negativos, junto a la curva de capacidad discriminante.


In [ ]:
# 6. Matrices de Confusion y Curvas ROC
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

for ax, (name, preds) in zip(axes, eval_predictions.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt=',d', cmap='Blues', cbar=False,
        xticklabels=['Legitimo', 'Amenaza'],
        yticklabels=['Legitimo', 'Amenaza'],
        ax=ax
    )
    f1_val = float(f1_score(y_test, preds, zero_division=0)) * 100.0
    ax.set_title(f'{name}\nF1-Score: {f1_val:.2f}%', fontsize=11, fontweight='bold')
    ax.set_xlabel('Clase Predicha')
    ax.set_ylabel('Clase Real')

plt.tight_layout()
plt.savefig('reports/figures/06_confusion_matrices.png', dpi=300)
plt.show()

# Curvas ROC
plt.figure(figsize=(7, 5))
for name, probs in eval_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = float(roc_auc_score(y_test, probs))
    plt.plot(fpr, tpr, lw=2, label=f'{name} - AUC: {auc_val:.3f}')

plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5)
plt.title('Figura 7. Curvas ROC Comparativas Multimodelo', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Tasa de Falsos Positivos - FPR')
plt.ylabel('Tasa de Verdaderos Positivos - TPR')
plt.legend(loc='lower right', frameon=True, facecolor='white')
plt.tight_layout()
plt.savefig('reports/figures/07_multi_model_roc.png', dpi=300)
plt.show()


## 7. Inteligencia Artificial Explicable - XAI: Importancia de Variables en el Arbol

El Arbol de Decision provee interpretabilidad nativa de caja blanca cuantificando el peso relativo que tuvo cada metrica en las bifurcaciones logicas.


In [ ]:
# 7. Ranking de Importancia de Caracteristicas
dt_model = models_dict['Arbol de Decision']
importances = pd.Series(dt_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 4.5))
sns.barplot(x=importances.values * 100, y=importances.index, palette='mako')
plt.title('Figura 8. Importancia Relativa de Variables en el Arbol de Decision - XAI', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Peso Predictivo Relativo (%)')
plt.ylabel('Variable de Entrada')

for idx, val in enumerate(importances.values * 100):
    plt.text(val + 0.5, idx, f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')

plt.xlim(0, max(importances.values * 100) * 1.15)
plt.tight_layout()
plt.savefig('reports/figures/08_feature_importances.png', dpi=300)
plt.show()

print('Desglose de Importancia de Variables:')
for feat_name, imp_val in importances.items():
    print(f' - {feat_name:18s}: {imp_val*100:6.2f}%')


## 8. Conclusiones y Justificacion Tecnica para la Sustentacion

### 1. Seleccion del Arbol de Decision - Modelo Ganador
- **Equilibrio Predictivo:** 73.71% de Exactitud y 72.16% de F1-Score general sobre datos de prueba.
- **Velocidad Ultrarrapida en Produccion:** Latencia inferior a 0.0001 milisegundos por correo, procesando cientos de peticiones por segundo sin sobrecargar el servidor.
- **Explicabilidad Total:** Alimenta de forma transparente el panel de decision en 4 niveles para el usuario final.

### 2. Descarte de K-Nearest Neighbors - KNN
- Aunque obtiene un F1-Score cercano de 72.82%, su latencia de inferencia es casi 100 veces mayor con 0.0093 milisegundos por consulta.
- Requiere computar distancias euclidianas contra mas de 73000 vectores en memoria RAM para cada clasificacion, resultando inviable en pasarelas corporativas con alta concurrencia.

### 3. Colapso de Gaussian Naive Bayes
- Colapso con un F1-Score critico de 1.98% y una sensibilidad de apenas 1.00%, dejando pasar el 99% de los ataques de phishing.
- Causa teorica: Violacion directa de sus asunciones matematicas. Naive Bayes asume independencia condicional entre variables y distribucion normal gaussiana. En ciberseguridad los enlaces y las palabras de urgencia estan fuertemente correlacionados y las variables binarias como has_url o has_ip tienen alta dispersion con inflacion de ceros.


In [ ]:
# 9. Serializacion de Modelos
joblib.dump(dt_model, 'models/decision_tree_model.joblib')
joblib.dump(scaler, 'models/robust_scaler.joblib')
print('[OK] Modelo Arbol de Decision y RobustScaler serializados en models/.')
